In [ ]:
from GetGapScore import gap_scores, gap_scores_with_time, zips_external, avg_gap_scores

In [ ]:
gap_scores = gap_scores[~gap_scores["gap_score"].isna()]
gap_scores.sort_values(by="gap_score", ascending=False)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.hist(gap_scores["gap_score"].dropna(), bins=30, color="skyblue", edgecolor="black")
plt.title("Distribution of Gap Scores")
plt.xlabel("Gap Score")
plt.ylabel("Number of Zipcodes")
plt.grid(True)
plt.show()

In [ ]:
avg_gap_scores.describe()

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(avg_gap_scores["gap_score"].dropna(), bins=30, color="skyblue", edgecolor="black")
plt.title("Distribution of Gap Scores")
plt.xlabel("Gap Score")
plt.ylabel("Number of Zipcodes")
plt.grid(True)
plt.show()

In [ ]:
gap_scores.info()

In [ ]:
gap_scores_with_time.info()

In [ ]:
avg_gap_scores.info()

In [ ]:
import pandas as pd
# Fix object-type columns by converting them to numeric, coercing errors to NaN
gap_scores_with_time["population"] = pd.to_numeric(gap_scores_with_time["population"], errors="coerce")
gap_scores_with_time["demand_score"] = pd.to_numeric(gap_scores_with_time["demand_score"], errors="coerce")
gap_scores_with_time["gap_score"] = pd.to_numeric(gap_scores_with_time["gap_score"], errors="coerce")


In [ ]:
from scipy.stats import linregress

def trend_test(df, zipcode):
    subset = df[df["zipcode"] == zipcode].sort_values("month")
    subset = subset.dropna(subset=["gap_score"])  # drop rows where gap_score is NaN

    if len(subset) < 3 or subset["gap_score"].nunique() == 1:
        return float("nan"), float("nan")  # insufficient data for trend

    months = pd.to_datetime(subset["month"]).map(pd.Timestamp.toordinal)
    result = linregress(months, subset["gap_score"])
    return result.slope, result.pvalue

# Apply to each zipcode with better handling
results = (
    gap_scores_with_time.groupby("zipcode", group_keys=False)
    .apply(lambda x: pd.Series(trend_test(x, x["zipcode"].iloc[0]), index=["slope", "pvalue"]))
    .reset_index()
)

# Optional: drop rows with NaN slope/pvalue
results = results.dropna(subset=["slope", "pvalue"])

# Sort and view
results[results["pvalue"] <= 0.05].sort_values("pvalue")

In [ ]:
import pymannkendall as mk

def mann_kendall_test(df, zipcode):
    subset = df[df["zipcode"] == zipcode].sort_values("month")
    subset = subset.dropna(subset=["gap_score"])
    
    if len(subset) < 3 or subset["gap_score"].nunique() == 1:
        return float("nan"), float("nan")
    
    result = mk.original_test(subset["gap_score"])
    return result.trend, result.p  # returns 'increasing', 'decreasing', or 'no trend'

# Apply to each ZIP
mk_results = (
    gap_scores_with_time.groupby("zipcode", group_keys=False)
    .apply(lambda x: pd.Series(mann_kendall_test(x, x["zipcode"].iloc[0]), index=["mk_trend", "mk_pvalue"]))
    .reset_index()
)

combined_results = pd.merge(results, mk_results, on="zipcode")
combined_results[(combined_results["pvalue"] <= 0.05) | ((combined_results["mk_trend"] != "no trend"))].sort_values("pvalue")

In [ ]:
increasing_zips = ["78214", "78260", "78237", "78009", "78064"]
decreasing_zips = ["78242", "78210", "78244", "78829", "78253", "78073", "78226", "78656", "78247"]

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

subset = gap_scores_with_time[gap_scores_with_time["zipcode"].isin(increasing_zips)].copy()
subset["month_dt"] = pd.to_datetime(subset["month"])

monthly_avg = (
    gap_scores_with_time
    .copy()
    .assign(month_dt=lambda df: pd.to_datetime(df["month"]))
    .groupby("month_dt")["gap_score"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(12, 6))
sns.lineplot(data=subset, x="month_dt", y="gap_score", hue="zipcode", marker="o")
sns.lineplot(data=monthly_avg, x="month_dt", y="gap_score", color="black", linestyle="--", label="Monthly Avg", linewidth=2) # Plot Average for all zips
plt.title("Gap Score Over Time by ZIP Code (Increasing)")
plt.xlabel("Month")
plt.ylabel("Gap Score")
plt.grid(True)
plt.legend(title="ZIP Code")
plt.tight_layout()
plt.show()


In [ ]:
subset = gap_scores_with_time[gap_scores_with_time["zipcode"].isin(decreasing_zips)].copy()
subset["month_dt"] = pd.to_datetime(subset["month"])

monthly_avg = (
    gap_scores_with_time
    .copy()
    .assign(month_dt=lambda df: pd.to_datetime(df["month"]))
    .groupby("month_dt")["gap_score"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(12, 6))
sns.lineplot(data=subset, x="month_dt", y="gap_score", hue="zipcode", marker="o")
sns.lineplot(data=monthly_avg, x="month_dt", y="gap_score", color="black", linestyle="--", label="Monthly Avg", linewidth=2) # Plot Average for all zips
plt.title("Gap Score Over Time by ZIP Code (Decreasing)")
plt.xlabel("Month")
plt.ylabel("Gap Score")
plt.grid(True)
plt.legend(title="ZIP Code")
plt.tight_layout()
plt.show()


In [ ]:
subset = gap_scores_with_time[gap_scores_with_time["zipcode"].isin(increasing_zips)].copy()
subset["month_dt"] = pd.to_datetime(subset["month"])

monthly_avg = (
    gap_scores_with_time
    .copy()
    .assign(month_dt=lambda df: pd.to_datetime(df["month"]))
    .groupby("month_dt")["demand_score"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(12, 6))
sns.lineplot(data=subset, x="month_dt", y="demand_score", hue="zipcode", marker="o")
sns.lineplot(data=monthly_avg, x="month_dt", y="demand_score", color="black", linestyle="--", label="Monthly Avg", linewidth=2) # Plot Average for all zips
plt.title("Demand Over Time by ZIP Code")
plt.xlabel("Month")
plt.ylabel("Demand Per 1000 People")
plt.grid(True)
plt.legend(title="ZIP Code")
plt.tight_layout()
plt.show()

In [ ]:
subset = gap_scores_with_time[gap_scores_with_time["zipcode"].isin(decreasing_zips)].copy()
subset["month_dt"] = pd.to_datetime(subset["month"])

monthly_avg = (
    gap_scores_with_time
    .copy()
    .assign(month_dt=lambda df: pd.to_datetime(df["month"]))
    .groupby("month_dt")["demand_score"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(12, 6))
sns.lineplot(data=subset, x="month_dt", y="demand_score", hue="zipcode", marker="o")
sns.lineplot(data=monthly_avg, x="month_dt", y="demand_score", color="black", linestyle="--", label="Monthly Avg", linewidth=2) # Plot Average for all zips
plt.title("Demand Over Time by ZIP Code")
plt.xlabel("Month")
plt.ylabel("Demand Per 1000 People")
plt.grid(True)
plt.legend(title="ZIP Code")
plt.tight_layout()
plt.show()

In [ ]:
high_scores = avg_gap_scores.sort_values(by="gap_score", ascending=False).head(7)
top7 = high_scores["zipcode"].tolist()

In [ ]:
subset = gap_scores_with_time[gap_scores_with_time["zipcode"].isin(top7)].copy()
subset["month_dt"] = pd.to_datetime(subset["month"])

monthly_avg = (
    gap_scores_with_time
    .copy()
    .assign(month_dt=lambda df: pd.to_datetime(df["month"]))
    .groupby("month_dt")["gap_score"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(12, 6))
sns.lineplot(data=subset, x="month_dt", y="gap_score", hue="zipcode", marker="o")
sns.lineplot(data=monthly_avg, x="month_dt", y="gap_score", color="black", linestyle="--", label="Monthly Avg", linewidth=2) # Plot Average for all zips
plt.title("Gap Score Over Time by ZIP Code (Top 7)")
plt.xlabel("Month")
plt.ylabel("Gap Score")
plt.grid(True)
plt.legend(title="ZIP Code")
plt.tight_layout()
plt.show()

In [ ]:
subset = gap_scores_with_time[gap_scores_with_time["zipcode"].isin(top7)].copy()
subset["month_dt"] = pd.to_datetime(subset["month"])

monthly_avg = (
    gap_scores_with_time
    .copy()
    .assign(month_dt=lambda df: pd.to_datetime(df["month"]))
    .groupby("month_dt")["demand_score"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(12, 6))
sns.lineplot(data=subset, x="month_dt", y="demand_score", hue="zipcode", marker="o")
sns.lineplot(data=monthly_avg, x="month_dt", y="demand_score", color="black", linestyle="--", label="Monthly Avg", linewidth=2) # Plot Average for all zips
plt.title("Demand Over Time by ZIP Code (Top 7)")
plt.xlabel("Month")
plt.ylabel("Demand Per 1000 People")
plt.grid(True)
plt.legend(title="ZIP Code")
plt.tight_layout()
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

sns.set_theme(style="whitegrid", context="talk")

# Prepare data
subset = gap_scores_with_time[gap_scores_with_time["zipcode"].isin(avg_gap_scores["zipcode"])].copy()
subset["month_dt"] = pd.to_datetime(subset["month"])

# Monthly average
monthly_avg = (
    gap_scores_with_time
    .copy()
    .assign(month_dt=lambda df: pd.to_datetime(df["month"]))
    .groupby("month_dt")["gap_score"]
    .mean()
    .reset_index()
)

# Interquartile range
monthly_bounds = (
    subset
    .groupby("month_dt")["gap_score"]
    .quantile([0.25, 0.75])
    .unstack()
    .rename(columns={0.25: "q25", 0.75: "q75"})
    .reset_index()
)

# Plot
plt.figure(figsize=(14, 7))

# Light gray lines for all ZIPs
sns.lineplot(
    data=subset[~subset["zipcode"].isin(top7)],
    x="month_dt", y="gap_score",
    hue="zipcode", palette="gray",
    alpha=0.2, linewidth=1, legend=False
)

# Define a color palette with 7 distinct colors
top7_palette = sns.color_palette("tab10", n_colors=len(top7))

# Plot top 7 ZIPs with different colors
for i, zip_code in enumerate(top7):
    sns.lineplot(
        data=subset[subset["zipcode"] == zip_code],
        x="month_dt", y="gap_score", linewidth=1.5,
        label=zip_code  # Use ZIP code as label
    )


# Shaded range
plt.fill_between(
    monthly_bounds["month_dt"],
    monthly_bounds["q25"],
    monthly_bounds["q75"],
    color="gray", alpha=0.3,
    label="25–75% Range"
)

# Monthly average line
sns.lineplot(
    data=monthly_avg,
    x="month_dt", y="gap_score",
    color="black", linestyle="--",
    label="Monthly Avg", linewidth=2
)

# Labels and legend
plt.xlabel("Month")
plt.ylabel("Gap Score")
plt.xticks(rotation=45)
plt.grid(True)

# Smaller legend in top right corner
plt.legend(
    loc="upper right",         # Top-left corner
    fontsize="x-small",       # Small legend labels
    title_fontsize="small",   # Small legend title
    frameon=True,
    framealpha=0.95
)


plt.tight_layout()
#plt.savefig("../Vizualizations/PencilGraph.png", dpi=1200, bbox_inches="tight")
plt.show()


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set(style="whitegrid", context="talk")

# Subset and process
subset = gap_scores_with_time[gap_scores_with_time["zipcode"].isin(increasing_zips)].copy()
subset["month_dt"] = pd.to_datetime(subset["month"])

monthly_avg = (
    gap_scores_with_time
    .assign(month_dt=lambda df: pd.to_datetime(df["month"]))
    .groupby("month_dt")["gap_score"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(9, 7))
sns.lineplot(
    data=subset,
    x="month_dt", y="gap_score",
    hue="zipcode",
    marker="o",
    linewidth=2
)

sns.lineplot(
    data=monthly_avg,
    x="month_dt", y="gap_score",
    color="black",
    linestyle="--",
    label="Monthly Avg",
    linewidth=3
)

plt.xlabel("Month")
plt.ylabel("Gap Score")
plt.xticks(rotation=45)

# ✅ Legend inside the plot (adjust `loc` as needed)
plt.legend(
    title="ZIP Code",
    loc="upper left",   # Try "upper right" or "lower left" if overlap happens
    frameon=True,
    fontsize=10,
    title_fontsize=12
)

plt.tight_layout()
#plt.savefig("../Visualizations/IncreasingZips.png", dpi=1200, bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Sort and select top 20 ZIP codes
top20_gap_scores = avg_gap_scores.sort_values(by="gap_score", ascending=False).head(20)

# Calculate overall average
overall_avg = avg_gap_scores["gap_score"].mean()

# Plot
plt.figure(figsize=(9, 7))
barplot = sns.barplot(
    data=top20_gap_scores,
    x="zipcode",
    y="gap_score",
    color="skyblue",
    edgecolor="black"
)

# Add exact values on top of bars
for index, row in top20_gap_scores.reset_index().iterrows():
    plt.text(
        index,
        row["gap_score"] + 0.01,  # Slightly above the bar
        f"{row['gap_score']:.2f}",
        ha="center",
        va="bottom",
        fontsize=9
    )

# Add average line
plt.axhline(
    overall_avg,
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Overall Avg ({overall_avg:.2f})"
)

# Titles and labels
# plt.title("Top 20 ZIP Codes by Average Monthly Gap Score")
plt.xlabel("ZIP Code")
plt.ylabel("Average Monthly Gap Score")
plt.xticks(rotation=45, fontsize=11)
plt.legend()
plt.tight_layout()
#plt.savefig("../Visualizations/Top25GapScores.png", dpi=1200, bbox_inches="tight")
plt.show()
